# Finite-temperature phonons with pheasy and machine-learned potentials

This notebook computes phonons at a finite temperature with the
`ForceFieldFiniteTemperaturePhononMaker` workflow. The forces come from three
MACE potentials, so no DFT is needed.

Section 1 runs five materials at 100, 300, 500, 700 and 900 K, in the 0 K cell.
Ca3Ir4Sn13 is worked through step by step. Section 2 runs the same materials in
the cell at each temperature, found with an NPT MD. Section 3 tests the length
of the MD and the number of snapshots for KNaICl.

This workflow is new. It is less tested than the other phonon workflows in
atomate2, and its defaults and output may still change.

## Background

Harmonic phonons from finite displacements describe the crystal at 0 K. Many
materials have imaginary modes at 0 K. The anharmonic motion of the atoms can
stabilize these modes at higher temperature. This workflow fits effective
harmonic force constants to an MD trajectory at the temperature of interest, as
in the temperature-dependent effective potential (TDEP) method of Hellman et
al., Phys. Rev. B 84, 180301 (2011). These force constants include the effect
of the anharmonic forces on the frequencies at that temperature. They give no
phonon lifetimes.

The flow has these steps:

1. The structure is relaxed. The relaxation can be switched off, as done below.
2. A diagonal supercell is built. Its lattice vectors are at least `min_length`
   long, 12 Å by default. A supercell matrix can also be given.
3. Optional, off by default: an NPT MD gives the cell at the temperature.
   Section 2 explains this step.
4. An NVT MD run of the supercell at the temperature. The defaults are 8 ps with
   a 1 fs time step and a Langevin thermostat.
5. The first 1 ps of the MD is left out. 50 snapshots are picked, evenly spread
   over the rest of the trajectory.
6. The forces on the snapshots and on the undisplaced supercell are computed.
7. pheasy fits the second-order force constants with LASSO. The displacements
   are measured from the undisplaced supercell. The forces of the undisplaced
   supercell are subtracted from the forces of each snapshot.
8. The band structure and the density of states (DOS) follow from the force
   constants. Imaginary modes are reported. They are not removed.

The MD trajectory is also checked. The check looks for melting and for a move
away from the starting structure. It also looks at the drift of the potential
energy late in the run. The result is stored in the output. A warning is raised
if the check fails, but the fit is still done.

Without the NPT step, the MD runs at constant volume (NVT). The volume is that
of the relaxed structure, or of the input structure when the relaxation is
switched off, as in this notebook. Thermal expansion is not included. The same
workflow runs with VASP as
`atomate2.vasp.flows.finite_temperature_phonons.FiniteTemperaturePhononMaker`.
`atomate2.forcefields.flows.finite_temperature_phonons` also has two mixed
versions. One runs the MD with VASP and the force calculations with a force
field. The other one does the reverse.

## Thermostats

The fit only uses the positions of the atoms in the MD, not how they move. The
snapshots have to follow the canonical distribution, so that every phonon mode
holds its share of the thermal energy.

The NVT MD uses a Langevin thermostat. It adds a friction and a random force to
every atom, and it samples the canonical distribution (Bussi and Parrinello,
Phys. Rev. E 75, 056707 (2007)). The friction is 10 ps^-1. A Nose-Hoover
thermostat acts on the whole crystal through one extra variable. It changes the
motion of the atoms less, which matters for vibrational spectra or diffusion.
But in a nearly harmonic crystal the modes barely exchange energy. For a
harmonic oscillator weakly coupled to a Nose-Hoover thermostat, the dynamics is
not ergodic (Legoll et al., Arch. Ration. Mech. Anal. 184, 449 (2007)). A
crystal is closest to harmonic at low temperature.

The NPT MD of Section 2 uses ASE's `MTKNPT`. It combines a Nose-Hoover
thermostat with one thermostat variable and the barostat of Martyna, Tobias and Klein, which samples the
isothermal-isobaric ensemble (J. Chem. Phys. 101, 4177 (1994)). Only its
average cell is used. With VASP, the NPT MD uses VASP's Langevin thermostat
and Parrinello-Rahman barostat, since VASP only runs NPT MD with these.

`thermostat="nose-hoover"` switches the NVT MD to a Nose-Hoover thermostat with
one thermostat variable. Section 3 compares the two for KNaICl. `random_seed`
seeds the initial velocities and the random forces of the Langevin thermostat.

## Installation

The `pheasy` and `ase` extras are needed, plus MACE and the Materials Project
client:

```
pip install 'atomate2[pheasy,ase]'
pip install 'mace-torch>=0.3.16' mp-api
```

The `pheasy` extra pulls in pheasy and ALM. ALM is compiled from source. The
hiPhive tutorial and the atomate2 VASP documentation list what the build needs.

## The potentials

The notebook uses three MACE foundation models. All three are released under the
Academic Software License.

- **MACE-OMAT-0-medium.** `mace_mp(model="medium-omat-0")` downloads it once and
  caches it under `~/.cache/mace`.
- **MACE-MATPES-PBE-0** and **MACE-MATPES-r2SCAN-0.** These are MACE-OMAT-0
  fine-tuned on the MatPES data set at the PBE and r2SCAN levels. Download the two files from the `mace_matpes_0` release of
  [mace-foundations](https://github.com/ACEsuit/mace-foundations/releases/tag/mace_matpes_0):
  - [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  - [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model)

Set the paths to the two downloaded files in the next cell. The Ca3Ir4Sn13
example below only needs MACE-OMAT-0-medium.

The MD takes most of the time, so a GPU helps a lot. The next cell uses one if
PyTorch finds it. All our runs used `float64` on one NVIDIA A100 GPU of a
Perlmutter node.

In [ ]:
import os
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
# hide the warnings of torch, e3nn and MACE when the models are loaded
warnings.filterwarnings("ignore", module=r"(torch|e3nn|mace)(\.|$)")

import torch  # noqa: E402
from mace.calculators import mace_mp  # noqa: E402

# Replace the two paths with the files you downloaded.
MODEL_FILES = {
    "OMAT-0-medium": "medium-omat-0",
    "MATPES-PBE-0": "/path/to/MACE-matpes-pbe-omat-ft.model",
    "MATPES-r2SCAN-0": "/path/to/MACE-matpes-r2scan-omat-ft.model",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def calc_kwargs(model: str) -> dict:
    """Return the calculator settings for one of the models above."""
    return {"model": MODEL_FILES[model], "device": DEVICE, "default_dtype": "float64"}


_ = mace_mp(**calc_kwargs("OMAT-0-medium"))  # downloads and caches on first use

## One calculator for all jobs

Every force field job builds its own calculator. `run_locally` runs all of
them in this one Python process, and the workflow has a little over 50 jobs.
Loading MACE for each of them grows the memory by about 150 MB per job. This
cell makes the jobs share one calculator, as in the thermal expansion tutorial.
It is only needed when many jobs run in one process, not with a workflow
manager.

In [ ]:
from ase.calculators.calculator import Calculator

import atomate2.forcefields.utils as ff_utils

_ase_calculator = ff_utils.ase_calculator
_calculators: dict[str, Calculator] = {}


def shared_ase_calculator(calculator_meta: object, **kwargs: object) -> Calculator:
    """Build each calculator once and reuse it for every job."""
    key = repr((calculator_meta, sorted(kwargs.items())))
    if key not in _calculators:
        _calculators[key] = _ase_calculator(calculator_meta, **kwargs)
    _calculators[key].reset()
    return _calculators[key]


ff_utils.ase_calculator = shared_ase_calculator

## 1. Five materials from 100 K to 900 K

The first part runs Ca3Ir4Sn13 at 300 K with MACE-OMAT-0-medium step by step.
The other temperatures, potentials and materials follow with the same code.

## The structure

Ca3Ir4Sn13 (mp-1200211) is cubic, with 40 atoms in the primitive cell. We start
from its PBEsol cell in the Materials Project's Harmonic Phonon Database. The
phonon documents of the Materials Project hold this cell. Set your key first with
`export MP_API_KEY=...`.

In [ ]:
from mp_api.client import MPRester
from pymatgen.core import Structure

if not os.environ.get("MP_API_KEY"):
    raise OSError("export MP_API_KEY before running this cell")


def get_pheasy_structure(mp_id: str) -> Structure:
    """Get the PBEsol cell of a material in the Harmonic Phonon Database."""
    with MPRester() as mpr:
        (doc,) = mpr.materials.phonon.search(
            material_ids=[mp_id],
            phonon_method="pheasy",
            fields=["structure"],
            num_chunks=1,
            chunk_size=10,
        )
    return doc.structure


structure = get_pheasy_structure("mp-1200211")
structure.lattice

## Building the workflow

`ForceFieldFiniteTemperaturePhononMaker.from_force_field_name` sets one force
field for the relaxation, the MD and the force calculations. Any MACE model
routes through `mace_mp`, and the model name in `calculator_kwargs` selects the
weights. The name "MACE-MP-0" only picks this calculator. The output keeps it as
`force_field_name`, and `force_field_kwargs` holds the model that was loaded.

We skip the relaxation with `relax_initial_structure=False`. The cell is
already relaxed with PBEsol, and our AIMD reference below uses the same cell. If
your input cell is not properly relaxed, keep the default
`relax_initial_structure=True`. The force field then relaxes the cell before
the MD.

We give the supercell matrix of our AIMD reference, 2x2x2 with 320 atoms.
Everything else stays at the workflow defaults.

In [ ]:
from atomate2.forcefields.flows.finite_temperature_phonons import (
    ForceFieldFiniteTemperaturePhononMaker,
)

maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
    "MACE-MP-0",
    calculator_kwargs=calc_kwargs("OMAT-0-medium"),
    relax_initial_structure=False,
    temperature=300,
)
flow = maker.make(structure, supercell_matrix=[[2, 0, 0], [0, 2, 0], [0, 0, 2]])
flow.draw_graph().show()

## Running the workflow

The flow has one MD job and 51 force calculations, 50 snapshots and the
undisplaced supercell. On one A100 GPU the whole run took 12 minutes. The MD
took 9 minutes, the force calculations 12 seconds and the fit 2 minutes.

`create_folders=True` is needed, because the snapshots are read from the
trajectory file in the folder of the MD job.

In [ ]:
from jobflow import JobStore, run_locally
from maggma.stores import MemoryStore

job_store = JobStore(MemoryStore(), additional_stores={"data": MemoryStore()})
responses = run_locally(flow, store=job_store, create_folders=True, ensure_success=True)
doc = responses[flow.output.uuid][1].output

## Results

The output is a `FiniteTemperaturePhononDoc`. It holds the band structure, the
DOS, the force constants, the fit and the trajectory check.

In [ ]:
import numpy as np

health = doc.trajectory_health
{
    "highest frequency in THz": float(np.max(doc.phonon_bandstructure.bands)),
    "imaginary modes on the band path": doc.has_imaginary_modes,
    "lowest frequency at the commensurate q-points in THz": doc.lowest_frequency,
    "LASSO penalty": doc.lasso_alpha,
    "force RMSE in eV/A": doc.force_rmse,
    "trajectory check": health.verdict,
    "Lindemann ratio": health.lindemann_ratio,
}

Our run gave a highest frequency of 5.97 THz, no imaginary modes and a stable
trajectory.

- `has_imaginary_modes` checks the whole band path. `lowest_frequency` only covers
  the q-points commensurate with the supercell. The fit sets the frequencies at
  these points directly. Everywhere else they are interpolated. Section 3 shows
  why the difference matters.
- The force RMSE is the part of the forces that the force constants do not
  describe. It grows with temperature, from 0.05 eV/Å at 100 K to 0.29 eV/Å at
  900 K for Ca3Ir4Sn13.
- The Lindemann ratio is the root mean square vibration of the atoms around
  their mean positions, divided by the nearest-neighbor distance. By the
  Lindemann criterion, a solid melts when this ratio reaches a fixed value
  (Lindemann, Phys. Z. 11, 609 (1910); Gilvarry, Phys. Rev. 102, 308 (1956)).
  The check reports the trajectory as melted above 0.15. This is the ratio at
  melting of fcc solids, defined the same way (Saija et al., J. Chem. Phys. 124,
  244504 (2006)).

## Comparing with 0 K

The Materials Project's Harmonic Phonon Database has the 0 K phonons of the same
cell, computed with PBEsol (Sahasrabuddhe et al., ChemRxiv (2026),
doi:10.26434/chemrxiv.15004632/v1). The next cell plots them next to ours. The
two band structures use different paths through the Brillouin zone, so they are
drawn side by side.

In [ ]:
import re

import matplotlib.pyplot as plt
from pymatgen.phonon.bandstructure import PhononBandStructureSymmLine
from pymatgen.phonon.dos import PhononDos


def plot_bands(
    ax: plt.Axes, bs: PhononBandStructureSymmLine, label: str | None = None, **style
) -> None:
    """Plot a phonon band structure, one segment of the path at a time."""
    distance = np.array(bs.distance)
    positions, names = [], []
    for idx, branch in enumerate(bs.branches):
        part = slice(branch["start_index"], branch["end_index"] + 1)
        lines = ax.plot(distance[part], bs.bands[:, part].T, **style)
        if idx == 0 and label:
            lines[0].set_label(label)
        start, end = branch["name"].split("-")
        if positions and np.isclose(positions[-1], distance[part.start]):
            if names[-1] != start:
                names[-1] += "|" + start
        else:
            positions.append(distance[part.start])
            names.append(start)
        positions.append(distance[branch["end_index"]])
        names.append(end)
    names = [name.replace("\\Gamma", "Γ").replace("GAMMA", "Γ") for name in names]
    names = [re.sub(r"_(\w+)", r"$_{\1}$", name) for name in names]
    ax.set_xticks(positions, names)
    ax.set_xlim(distance[0], distance[-1])
    ax.axhline(0, color="gray", lw=0.5)


def plot_dos(ax: plt.Axes, dos: PhononDos, label: str | None = None, **style) -> None:
    """Plot a phonon DOS sideways, to sit next to a band structure."""
    ax.plot(dos.densities, dos.frequencies, label=label, **style)
    ax.set_xlabel("DOS")


with MPRester() as mpr:
    bs_0k = mpr.materials.phonon.get_bandstructure_from_material_id(
        "mp-1200211", phonon_method="pheasy"
    ).to_pmg
    dos_0k = mpr.materials.phonon.get_dos_from_material_id(
        "mp-1200211", phonon_method="pheasy"
    ).to_pmg

fig, axes = plt.subplots(
    1, 3, figsize=(13, 4), sharey=True, gridspec_kw={"width_ratios": [3, 3, 1]}
)
plot_bands(axes[0], bs_0k, color="gray", lw=0.8)
plot_bands(axes[1], doc.phonon_bandstructure, color="C0", lw=0.8)
plot_dos(axes[2], dos_0k, label="0 K", color="gray")
plot_dos(axes[2], doc.phonon_dos, label="300 K", color="C0")
axes[0].set_title("0 K, PBEsol")
axes[1].set_title("300 K, MACE-OMAT-0-medium")
axes[0].set_ylabel("Frequency (THz)")
axes[2].legend()
plt.show()

At 0 K, Ca3Ir4Sn13 has imaginary modes down to about -0.75 THz. At 300 K they
are gone. The motion of the atoms at 300 K stabilizes them.

To check the force field, we compare with effective harmonic force constants
fitted to ab initio MD at 300 K. These were computed with PBEsol in the same
cell and supercell. Their highest frequency is 6.03 THz, against 5.97 THz from
MACE-OMAT-0-medium. Over the whole band path the frequencies differ by 0.07 THz
on average.

## The other temperatures, potentials and materials

The same workflow runs at five temperatures, with all three potentials and for
four more materials. AlAsPt5, Zn3P2 and Er5Tl3 also have imaginary modes at 0 K.
Bi4S3N2 is polar. The Materials Project lists AlAsPt5 and Bi4S3N2 as
theoretical. They are predicted compounds with no known experimental structure.

The function below builds and runs one workflow in its own folder. Sections 2
and 3 use it as well.

The loop runs 75 workflows, so it is off by default. Set `RUN_ALL = True` to
run it. Each of our runs took 6 to 22 minutes on one A100 GPU.

In [ ]:
from atomate2.common.schemas.finite_temperature_phonons import (
    FiniteTemperaturePhononDoc,
)

# mp-id and the diagonal of the supercell matrix of our AIMD reference
MATERIALS = {
    "Ca3Ir4Sn13": ("mp-1200211", (2, 2, 2)),
    "AlAsPt5": ("mp-1025306", (4, 4, 2)),
    "Zn3P2": ("mp-2071", (2, 2, 2)),
    "Bi4S3N2": ("mp-1245549", (2, 2, 2)),
    "Er5Tl3": ("mp-1105965", (2, 2, 2)),
    "KNaICl": ("mp-1002081", (3, 3, 2)),
}
TEMPERATURES = (100, 300, 500, 700, 900)


def run_ft(
    structure: Structure,
    supercell: tuple[int, int, int],
    model: str,
    root_dir: str,
    temperature: float,
    md_time: float = 8.0,
    n_snapshots: int = 50,
    relax: bool = False,
    run_npt: bool = False,
) -> FiniteTemperaturePhononDoc:
    """Run the finite-temperature phonon workflow and return its output."""
    maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
        "MACE-MP-0",
        calculator_kwargs=calc_kwargs(model),
        relax_initial_structure=relax,
        run_npt=run_npt,
        temperature=temperature,
        md_time=md_time,
        n_snapshots=n_snapshots,
    )
    flow = maker.make(structure, supercell_matrix=np.diag(supercell).tolist())
    os.makedirs(root_dir, exist_ok=True)
    responses = run_locally(
        flow,
        store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
        create_folders=True,
        root_dir=root_dir,
        ensure_success=True,
    )
    return responses[flow.output.uuid][1].output


RUN_ALL = False
docs = {}
if RUN_ALL:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        mp_id, supercell = MATERIALS[name]
        unit_cell = get_pheasy_structure(mp_id)
        for model in MODEL_FILES:
            for temperature in TEMPERATURES:
                docs[name, model, temperature] = run_ft(
                    unit_cell,
                    supercell,
                    model,
                    f"runs/{name}_{model}_{temperature}K",
                    temperature,
                )

The next cell plots the results of the loop. For each potential it shows the
band structure at 100, 500 and 900 K. Ca3Ir4Sn13 and Zn3P2 have 40 atoms in the
primitive cell, so their 120 branches are hard to follow. For these two the DOS
at all five temperatures is shown as well.

In [ ]:
def plot_temperatures(name: str, with_dos: bool) -> None:
    """Plot the band structures of one material at 100, 500 and 900 K."""
    ratios = [3, 1] * len(MODEL_FILES) if with_dos else [3] * len(MODEL_FILES)
    fig, axes = plt.subplots(
        1,
        len(ratios),
        figsize=(4.5 * len(MODEL_FILES), 4),
        sharey=True,
        gridspec_kw={"width_ratios": ratios},
    )
    colors = dict(
        zip(TEMPERATURES, plt.cm.plasma(np.linspace(0, 0.85, 5)), strict=True)
    )
    step = 2 if with_dos else 1
    for idx, model in enumerate(MODEL_FILES):
        ax = axes[step * idx]
        for temperature in (100, 500, 900):
            plot_bands(
                ax,
                docs[name, model, temperature].phonon_bandstructure,
                label=f"{temperature} K",
                color=colors[temperature],
                lw=0.6,
            )
        ax.set_title(model)
        if with_dos:
            for temperature in TEMPERATURES:
                plot_dos(
                    axes[step * idx + 1],
                    docs[name, model, temperature].phonon_dos,
                    label=f"{temperature} K",
                    color=colors[temperature],
                )
    axes[0].set_ylabel("Frequency (THz)")
    axes[-1].legend(fontsize=8)
    fig.suptitle(name, y=1.02)
    plt.show()


if RUN_ALL:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        plot_temperatures(name, with_dos=name in ("Ca3Ir4Sn13", "Zn3P2"))

## Our results

The next cell holds the results of our runs. For each material it gives the
highest frequency on the band path at each temperature. The dashed line is the
highest frequency of the AIMD reference at 300 K.

In [ ]:
import pandas as pd

# highest frequency on the band path in THz, at 100, 300, 500, 700 and 900 K
MAX_FREQUENCY = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (6.03, 5.97, 5.94, 5.89, 5.78),
        "MATPES-PBE-0": (5.80, 5.75, 5.74, 5.66, 5.64),
        "MATPES-r2SCAN-0": (6.03, 5.95, 5.91, 5.89, 5.78),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (7.80, 7.76, 7.70, 7.77, 7.78),
        "MATPES-PBE-0": (7.58, 7.66, 7.68, 7.67, 7.80),
        "MATPES-r2SCAN-0": (7.65, 7.58, 7.54, 7.68, 7.58),
    },
    "Zn3P2": {
        "OMAT-0-medium": (10.73, 10.44, 10.05, 9.80, 9.42),
        "MATPES-PBE-0": (10.42, 10.08, 9.91, 9.63, 9.27),
        "MATPES-r2SCAN-0": (11.26, 10.91, 10.69, 10.33, 10.14),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (17.35, 16.93, 15.98, 15.85, 15.12),
        "MATPES-PBE-0": (15.64, 15.21, 15.28, 14.56, 14.53),
        "MATPES-r2SCAN-0": (17.60, 17.23, 16.52, 16.00, 15.73),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (4.36, 4.25, 4.28, 4.24, 4.19),
        "MATPES-PBE-0": (4.15, 4.12, 4.05, 3.91, 3.86),
        "MATPES-r2SCAN-0": (4.18, 4.14, 4.15, 4.11, 3.98),
    },
}
# AIMD reference at 300 K: highest frequency in THz, and the mean absolute
# difference of each potential from it over all branches of the band path
AIMD_MAX_300K = {
    "Ca3Ir4Sn13": 6.03,
    "AlAsPt5": 7.79,
    "Zn3P2": 10.09,
    "Bi4S3N2": 17.03,
    "Er5Tl3": 3.97,
}
MAE_300K = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": 0.07,
        "MATPES-PBE-0": 0.17,
        "MATPES-r2SCAN-0": 0.10,
    },
    "AlAsPt5": {"OMAT-0-medium": 0.11, "MATPES-PBE-0": 0.19, "MATPES-r2SCAN-0": 0.17},
    "Zn3P2": {"OMAT-0-medium": 0.21, "MATPES-PBE-0": 0.10, "MATPES-r2SCAN-0": 0.44},
    "Bi4S3N2": {"OMAT-0-medium": 0.15, "MATPES-PBE-0": 0.46, "MATPES-r2SCAN-0": 0.16},
    "Er5Tl3": {"OMAT-0-medium": 0.18, "MATPES-PBE-0": 0.17, "MATPES-r2SCAN-0": 0.13},
}
# runs with imaginary modes on the band path, and runs whose trajectory check
# was not "stable"
IMAGINARY = {
    ("AlAsPt5", "MATPES-PBE-0"): (100,),
    ("Bi4S3N2", "OMAT-0-medium"): (100, 300, 500),
    ("Bi4S3N2", "MATPES-PBE-0"): (300, 500, 700, 900),
    ("Bi4S3N2", "MATPES-r2SCAN-0"): (100, 300, 500, 700, 900),
}
NOT_STABLE = {
    ("Ca3Ir4Sn13", "OMAT-0-medium", 100): "shifted_or_diffusing",
    ("Ca3Ir4Sn13", "MATPES-r2SCAN-0", 100): "shifted_or_diffusing",
    ("Er5Tl3", "MATPES-r2SCAN-0", 100): "shifted_or_diffusing",
    ("Zn3P2", "OMAT-0-medium", 900): "melted",
    ("Zn3P2", "MATPES-PBE-0", 900): "melted",
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, (name, by_model) in zip(axes, MAX_FREQUENCY.items(), strict=True):
    for model, values in by_model.items():
        ax.plot(TEMPERATURES, values, "o-", label=model)
    ax.axhline(AIMD_MAX_300K[name], color="black", ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Highest frequency (THz)")
fig.tight_layout()
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0),
)
plt.show()

pd.DataFrame(MAE_300K).T.assign(**{"AIMD highest frequency": pd.Series(AIMD_MAX_300K)})

The table gives the mean absolute difference from the AIMD reference at 300 K
in THz, over all branches and q-points of the band path.

- No potential is closest to AIMD for all five materials. MACE-OMAT-0-medium is
  closest for Ca3Ir4Sn13, AlAsPt5 and Bi4S3N2, within 0.07 to 0.15 THz.
  MACE-MATPES-r2SCAN-0 is closest for Er5Tl3 and MACE-MATPES-PBE-0 for Zn3P2.
- MACE-MATPES-PBE-0 gives the lowest frequencies for most materials. For
  Bi4S3N2 its highest branches lie about 1.8 THz below AIMD.
- Most frequencies soften with temperature. The highest frequency of Bi4S3N2
  drops by 2.2 THz from 100 K to 900 K with MACE-OMAT-0-medium. AlAsPt5 barely
  changes.
- The imaginary modes at 0 K are gone at every temperature for Ca3Ir4Sn13, Zn3P2
  and Er5Tl3. AlAsPt5 keeps a small one, -0.12 THz, with MACE-MATPES-PBE-0 at
  100 K. Bi4S3N2 keeps imaginary modes down to -0.8 THz in 12 of 15 runs. Its
  AIMD reference has a small one as well, at -0.12 THz. Bi4S3N2 is polar, so
  these runs and the AIMD reference are compared without the non-analytical
  correction. The three MACE potentials give no Born charges.

The trajectory check did not report "stable" for five runs:

- By the Lindemann criterion, Zn3P2 melts at 900 K with MACE-OMAT-0-medium and
  MACE-MATPES-PBE-0. Its Lindemann ratio there is 0.161 and 0.157. With
  MACE-MATPES-r2SCAN-0 it reaches 0.144 at 900 K, below the limit of 0.15. No
  other material reaches the limit up to 900 K. The highest ratio among them is
  0.127, for Bi4S3N2. Every run keeps the volume of the 0 K cell. Section 2 adds
  thermal expansion.
- At 100 K three runs are reported as shifted or diffusing: Ca3Ir4Sn13 with
  MACE-OMAT-0-medium and MACE-MATPES-r2SCAN-0, and Er5Tl3 with
  MACE-MATPES-r2SCAN-0. The mean positions of the atoms moved away from the
  starting structure by more than they vibrate. Both materials have imaginary
  modes at 0 K. Section 2 shows the same at 100 K in the force field's own cell,
  for Ca3Ir4Sn13 and AlAsPt5. From 300 K on, their NVT runs are stable.
  Most likely the atoms stay in a slightly distorted structure at 100 K and
  move between such structures at higher temperature. We did not check which
  distortion it is. The force constants of these runs describe vibrations
  around the symmetric structure, so use them with care.

## 2. Thermal expansion with an NPT MD

In Section 1 every MD ran in the PBEsol cell at 0 K. Two things change this
cell. Each force field has its own 0 K cell, a little different from the PBEsol
one. And the crystal expands as it heats up. With `run_npt=True` the workflow
includes both:

1. The force field relaxes the structure. Keep the default
   `relax_initial_structure=True`, so that the NPT MD starts from the force
   field's own 0 K cell.
2. An NPT MD of the supercell runs at the temperature and at `pressure`, 0 kbar
   by default. It runs for `npt_time`, 8 ps by default, with a Nose-Hoover
   thermostat and a Martyna-Tobias-Klein barostat.
3. The cell is averaged over the NPT MD after `npt_equilibration_time`, 2 ps by
   default. The average keeps the symmetry of the structure. The atoms keep
   their fractional coordinates and are then relaxed in the new cell, with the
   cell fixed.
4. The NVT MD, the snapshots, the fit and the trajectory check follow as in
   Section 1, in the new cell. The relaxed structure in the new cell is the
   undisplaced supercell of the fit.

The NPT trajectory is checked like the NVT one, and the result is stored as
`npt_trajectory_health`. `structure` holds the cell of the NVT MD and
`npt_input_structure` the cell that the NPT MD started from.

We run Ca3Ir4Sn13 at 900 K with MACE-OMAT-0-medium.

In [ ]:
maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
    "MACE-MP-0",
    calculator_kwargs=calc_kwargs("OMAT-0-medium"),
    run_npt=True,
    temperature=900,
)
flow = maker.make(structure, supercell_matrix=[[2, 0, 0], [0, 2, 0], [0, 0, 2]])
responses = run_locally(
    flow,
    store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
    create_folders=True,
    ensure_success=True,
)
doc_npt = responses[flow.output.uuid][1].output
{
    "lattice at 0 K in A": doc_npt.npt_input_structure.lattice.abc,
    "lattice at 900 K in A": doc_npt.structure.lattice.abc,
    "volume ratio": doc_npt.structure.volume / doc_npt.npt_input_structure.volume,
    "NPT trajectory check": doc_npt.npt_trajectory_health.verdict,
    "highest frequency in THz": float(np.max(doc_npt.phonon_bandstructure.bands)),
    "imaginary modes on the band path": doc_npt.has_imaginary_modes,
    "trajectory check": doc_npt.trajectory_health.verdict,
}

Our run took 19 minutes. The NPT MD and the NVT MD took about 8 minutes each
and the fit 2 minutes. The 0 K cell of MACE-OMAT-0-medium has a lattice
parameter of 9.825 Å, and its volume is 3.8% larger than that of the PBEsol
cell. At 900 K the lattice parameter is 9.964 Å, 4.3% more volume. Both
trajectories were stable. The highest frequency is 4.91 THz, against 5.78 THz
in the PBEsol cell in Section 1.

## The other temperatures, potentials and materials

We ran all 75 settings of Section 1 again with `run_npt=True`. To tell the two
effects apart, we also ran them with the relaxation but without the NPT MD,
that is in the force field's own 0 K cell. `run_ft` takes both options. The two
loops run 150 workflows, so they are off by default. Each of our runs took 6 to
29 minutes on one A100 GPU.

In [ ]:
RUN_NPT = False
if RUN_NPT:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        mp_id, supercell = MATERIALS[name]
        unit_cell = get_pheasy_structure(mp_id)
        for model in MODEL_FILES:
            for temperature in TEMPERATURES:
                for label, run_npt in (("relaxed", False), ("npt", True)):
                    docs[name, model, temperature, label] = run_ft(
                        unit_cell,
                        supercell,
                        model,
                        f"runs/{name}_{model}_{temperature}K_{label}",
                        temperature,
                        relax=True,
                        run_npt=run_npt,
                    )

## Our results

The next cell holds the results of our runs. The first plot gives the volume
relative to the PBEsol cell. The point at 0 K is the force field's relaxed cell.
The second plot gives the highest frequency for each potential in three cells:
the PBEsol cell of Section 1, the force field's 0 K cell, and the cell from the
NPT MD.

In [ ]:
# volume relative to the PBEsol cell: the force field's relaxed 0 K cell, then
# the NPT cell at 100, 300, 500, 700 and 900 K
VOLUME = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (1.0382, 1.0420, 1.0517, 1.0612, 1.0716, 1.0828),
        "MATPES-PBE-0": (1.0432, 1.0475, 1.0568, 1.0665, 1.0774, 1.0890),
        "MATPES-r2SCAN-0": (1.0192, 1.0227, 1.0311, 1.0398, 1.0491, 1.0591),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (1.0341, 1.0382, 1.0463, 1.0551, 1.0642, 1.0744),
        "MATPES-PBE-0": (1.0175, 1.0229, 1.0313, 1.0400, 1.0501, 1.0501),
        "MATPES-r2SCAN-0": (1.0021, 1.0061, 1.0133, 1.0210, 1.0302, 1.0392),
    },
    "Zn3P2": {
        "OMAT-0-medium": (1.0533, 1.0585, 1.0681, 1.0780, 1.0884, 1.1012),
        "MATPES-PBE-0": (1.0510, 1.0551, 1.0636, 1.0723, 1.0813, 1.0934),
        "MATPES-r2SCAN-0": (1.0087, 1.0125, 1.0211, 1.0301, 1.0387, 1.0491),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (1.1075, 1.1153, 1.1269, 1.1424, 1.1515, 1.1741),
        "MATPES-PBE-0": (1.0457, 1.0594, 1.0764, 1.0967, 1.1515, 1.1625),
        "MATPES-r2SCAN-0": (1.0292, 1.0381, 1.0502, 1.0656, 1.0807, 1.0861),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (1.0291, 1.0356, 1.0465, 1.0573, 1.0684, 1.0801),
        "MATPES-PBE-0": (1.1224, 1.1283, 1.1431, 1.1589, 1.1744, 1.1939),
        "MATPES-r2SCAN-0": (1.0646, 1.0697, 1.0806, 1.0925, 1.1034, 1.1164),
    },
}
# highest frequency on the band path in THz, at 100, 300, 500, 700 and 900 K,
# in the force field's relaxed 0 K cell
MAX_RELAXED = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (5.58, 5.58, 5.51, 5.47, 5.40),
        "MATPES-PBE-0": (5.37, 5.31, 5.26, 5.19, 5.17),
        "MATPES-r2SCAN-0": (5.78, 5.76, 5.69, 5.59, 5.54),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (7.19, 7.07, 7.09, 7.20, 7.17),
        "MATPES-PBE-0": (7.23, 7.24, 7.11, 7.09, 7.30),
        "MATPES-r2SCAN-0": (7.52, 7.43, 7.42, 7.49, 7.45),
    },
    "Zn3P2": {
        "OMAT-0-medium": (10.25, 9.82, 9.67, 9.39, 8.91),
        "MATPES-PBE-0": (9.99, 9.73, 9.35, 9.06, 8.70),
        "MATPES-r2SCAN-0": (11.19, 10.92, 10.46, 10.40, 9.84),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (16.74, 15.80, 15.47, 14.80, 14.41),
        "MATPES-PBE-0": (15.40, 15.02, 14.32, 14.55, 14.42),
        "MATPES-r2SCAN-0": (17.27, 16.95, 16.22, 15.74, 15.42),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (4.10, 4.07, 4.06, 4.00, 4.01),
        "MATPES-PBE-0": (3.70, 3.62, 3.56, 3.57, 3.40),
        "MATPES-r2SCAN-0": (3.83, 3.83, 3.77, 3.78, 3.72),
    },
}
# the same in the NPT cell
MAX_NPT = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (5.56, 5.39, 5.26, 5.11, 4.91),
        "MATPES-PBE-0": (5.33, 5.19, 5.03, 4.89, 4.71),
        "MATPES-r2SCAN-0": (5.72, 5.59, 5.44, 5.26, 5.06),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (7.18, 6.90, 6.90, 6.51, 6.88),
        "MATPES-PBE-0": (7.15, 6.94, 6.90, 6.63, 6.64),
        "MATPES-r2SCAN-0": (7.39, 7.27, 7.26, 7.11, 6.77),
    },
    "Zn3P2": {
        "OMAT-0-medium": (10.23, 9.82, 9.31, 8.91, 8.42),
        "MATPES-PBE-0": (9.97, 9.68, 9.19, 8.86, 8.19),
        "MATPES-r2SCAN-0": (11.16, 10.72, 10.39, 9.87, 9.66),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (16.39, 15.58, 15.15, 13.75, 13.63),
        "MATPES-PBE-0": (15.50, 15.04, 14.74, 13.39, 12.34),
        "MATPES-r2SCAN-0": (17.38, 16.63, 16.28, 15.33, 14.67),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (4.04, 3.99, 3.79, 3.77, 3.66),
        "MATPES-PBE-0": (3.67, 3.56, 3.46, 3.32, 3.25),
        "MATPES-r2SCAN-0": (3.78, 3.72, 3.64, 3.56, 3.44),
    },
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, (name, by_model) in zip(axes, VOLUME.items(), strict=True):
    for model, values in by_model.items():
        ax.plot((0, *TEMPERATURES), values, "o-", label=model)
    ax.axhline(1, color="black", ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Volume / PBEsol volume")
fig.tight_layout()
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0),
)
plt.show()

fig, axes = plt.subplots(3, 5, figsize=(16, 8.5), sharex=True)
for col, name in enumerate(VOLUME):
    for row, model in enumerate(MODEL_FILES):
        ax = axes[row, col]
        for label, values, style in (
            ("PBEsol cell", MAX_FREQUENCY, "o-"),
            ("relaxed 0 K cell", MAX_RELAXED, "s--"),
            ("NPT cell", MAX_NPT, "^:"),
        ):
            ax.plot(TEMPERATURES, values[name][model], style, label=label)
        if row == 0:
            ax.set_title(name)
        if col == 0:
            ax.set_ylabel(f"{model}\nHighest frequency (THz)")
        if row == 2:
            ax.set_xlabel("Temperature (K)")
fig.tight_layout()
fig.legend(
    *axes[0, 0].get_legend_handles_labels(),
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0),
)
plt.show()

We do not compare these runs with the AIMD reference. It used the PBEsol cell
at every temperature.

- The 0 K cells of the force fields are larger than the PBEsol cell, by up to
  12% in volume. MACE-OMAT-0-medium and MACE-MATPES-PBE-0 are trained on PBE
  data, and PBE gives larger cells than PBEsol. MACE-MATPES-r2SCAN-0 gives the
  cell closest to PBEsol for four of the five materials. For AlAsPt5 the two
  differ by 0.2%.
- From 0 K to 900 K the volume grows by 3 to 6.5%. Bi4S3N2 with
  MACE-MATPES-PBE-0 is the exception, with 11%.
- The larger 0 K cell alone lowers the frequencies. At 300 K the highest
  frequency of Ca3Ir4Sn13 drops from 5.97 THz in the PBEsol cell to 5.58 THz
  in the cell of MACE-OMAT-0-medium. With MACE-MATPES-PBE-0 the 0 K cell of
  Er5Tl3 is 12% larger, and its highest frequency at 300 K drops from 4.12 to
  3.62 THz. With MACE-MATPES-r2SCAN-0 the cell of Zn3P2 grows by less than 1%,
  and its highest frequency at 300 K stays at 10.9 THz.
- Thermal expansion lowers the frequencies further. At 100 K the NPT cell
  changes the highest frequency by 0.15 THz or less, except for Bi4S3N2. At
  900 K it lowers it by 0.15 to 0.8 THz. For Bi4S3N2 with MACE-MATPES-PBE-0,
  whose cell grows by 11%, it lowers it by 2.1 THz.
- At constant volume a frequency can also rise with temperature. In the PBEsol
  cell, the highest frequency of AlAsPt5 with MACE-MATPES-PBE-0 rises from
  7.58 THz at 100 K to 7.80 THz at 900 K. In the NPT cell it falls from 7.15 to
  6.64 THz.

The trajectory check did not report "stable" for these runs:

- Zn3P2 at 900 K melts with all three potentials in the NPT MD and in the NVT MD
  that follows. With MACE-MATPES-r2SCAN-0, its Lindemann ratio in Section 1
  stayed below the limit, at 0.144. In the NVT MD in the NPT cell it is 0.150,
  just above the limit. In the 0 K cell it melts at 900 K with MACE-OMAT-0-medium and
  MACE-MATPES-PBE-0.
- Bi4S3N2 melts at 900 K in the NPT cell. The NPT MD is reported as melted with
  all three potentials, and the NVT MD that follows with MACE-OMAT-0-medium and
  MACE-MATPES-PBE-0. With MACE-MATPES-PBE-0 the NPT MD is also reported as
  melted at 500 K and as shifted at 700 K, and the volume jumps by 5% between
  these two temperatures. In the PBEsol cell and in the 0 K cell none of its
  runs melted.
- AlAsPt5 at 900 K with MACE-MATPES-PBE-0: the NPT MD is reported as shifted,
  and its volume does not grow from 700 to 900 K. The cell of this run is not
  reliable, even though the NVT MD that follows is stable.
- At 100 K, the shift check fires for Ca3Ir4Sn13 and AlAsPt5. In the 0 K cell
  this happens with MACE-OMAT-0-medium and MACE-MATPES-r2SCAN-0 for both. In the
  NPT cell it happens for Ca3Ir4Sn13 with MACE-MATPES-r2SCAN-0 and for AlAsPt5
  with all three potentials. Section 1 discusses this.

Bi4S3N2 has imaginary modes on the band path in all 30 runs. A few runs at
100 K have small ones too: Ca3Ir4Sn13 with MACE-MATPES-r2SCAN-0 in both cells,
and AlAsPt5 in the NPT cell with MACE-OMAT-0-medium and MACE-MATPES-PBE-0. The
check reports all of these runs as shifted.

A run with the NPT MD took 12 to 29 minutes, against 6 to 21 minutes in the
0 K cell. To compare a force field with DFT phonons in a given cell, run in
that cell, as in Section 1. To predict the phonons at a temperature with the
force field alone, use `run_npt=True`.

## 3. MD length and number of snapshots for KNaICl

Two settings control how well the MD samples the temperature:

- `md_time` sets the length of the MD in ps. The default is 8 ps, of which the
  first 1 ps is left out.
- `n_snapshots` sets the number of snapshots used in the fit. The default is 50.

We tested 4, 8 and 16 ps with 25, 50 and 100 snapshots at 300 K for KNaICl
(mp-1002081). It has 4 atoms in the primitive cell and 72 in the 3x3x2
supercell. Like AlAsPt5 and Bi4S3N2, KNaICl is a predicted compound. KNaICl is polar, so it is compared with the AIMD reference without
the non-analytical correction. The highest frequency of that reference is
5.51 THz.

The grid runs 27 workflows, so it is off by default.

In [ ]:
RUN_GRID = False
if RUN_GRID:
    mp_id, supercell = MATERIALS["KNaICl"]
    knaicl = get_pheasy_structure(mp_id)
    for model in MODEL_FILES:
        for md_time in (4, 8, 16):
            for n_snapshots in (25, 50, 100):
                docs["KNaICl", model, md_time, n_snapshots] = run_ft(
                    knaicl,
                    supercell,
                    model,
                    f"runs/KNaICl_{model}_{md_time}ps_{n_snapshots}",
                    300,
                    md_time,
                    n_snapshots,
                )

In [ ]:
MD_TIMES = (4, 8, 16)
N_SNAPSHOTS = (25, 50, 100)
# by potential, rows 4, 8 and 16 ps, columns 25, 50 and 100 snapshots
MAE_GRID = {  # mean absolute difference from AIMD at 300 K in THz
    "OMAT-0-medium": (
        (0.179, 0.181, 0.147),
        (0.167, 0.170, 0.148),
        (0.162, 0.171, 0.158),
    ),
    "MATPES-PBE-0": (
        (0.158, 0.137, 0.133),
        (0.129, 0.108, 0.120),
        (0.142, 0.089, 0.105),
    ),
    "MATPES-r2SCAN-0": (
        (0.188, 0.161, 0.162),
        (0.191, 0.184, 0.136),
        (0.152, 0.154, 0.169),
    ),
}
MAX_GRID = {  # highest frequency on the band path in THz
    "OMAT-0-medium": ((5.48, 5.52, 5.66), (5.35, 5.56, 5.63), (5.57, 5.62, 5.61)),
    "MATPES-PBE-0": ((5.20, 5.18, 5.13), (5.07, 5.22, 5.33), (5.05, 5.31, 5.24)),
    "MATPES-r2SCAN-0": ((5.64, 5.51, 5.57), (5.32, 5.58, 5.42), (5.42, 5.53, 5.53)),
}
IMAGINARY_GRID = {  # True if the band path has imaginary modes
    "OMAT-0-medium": ((False, True, True), (True, True, False), (True, True, False)),
    "MATPES-PBE-0": ((True, True, True), (False, True, False), (False, False, True)),
    "MATPES-r2SCAN-0": ((True, True, True), (True, True, False), (True, True, True)),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, model in zip(axes, MODEL_FILES, strict=True):
    grid = np.array(MAE_GRID[model])
    image = ax.imshow(grid, cmap="viridis", vmin=0.08, vmax=0.20)
    for i in range(3):
        for j in range(3):
            mark = " *" if IMAGINARY_GRID[model][i][j] else ""
            ax.text(
                j, i, f"{grid[i, j]:.3f}{mark}", ha="center", va="center", color="w"
            )
    ax.set_xticks(range(3), N_SNAPSHOTS)
    ax.set_yticks(range(3), [f"{t} ps" for t in MD_TIMES])
    ax.set_xlabel("Number of snapshots")
    ax.set_title(model)
fig.colorbar(image, ax=axes, label="Mean absolute difference from AIMD (THz)")
plt.show()

pd.concat(
    {
        model: pd.DataFrame(
            MAX_GRID[model], index=[f"{t} ps" for t in MD_TIMES], columns=N_SNAPSHOTS
        )
        for model in MODEL_FILES
    },
    axis=1,
)

In the plot, a star marks a run with imaginary modes on the band path. The
table gives the highest frequency in THz.

- The mean absolute difference from AIMD lies between 0.09 and 0.19 THz. It
  shows no trend with the MD length or the number of snapshots. Each run is one
  MD trajectory, so these differences are the noise of a single run.
- The highest frequency changes by up to 0.3 THz between runs, also without a
  trend.
- 19 of the 27 runs have imaginary modes on the band path, down to -1.5 THz. In
  all but two of them they only appear at q-points that are not commensurate
  with the supercell, where the frequencies are interpolated from the fitted
  force constants. The AIMD reference has no imaginary modes in the same
  supercell. They appear in 6 of 9 runs with 25 snapshots, 8 of 9 with 50 and
  5 of 9 with 100. In an earlier round of the same grid with a Nose-Hoover
  thermostat, 12 of the 27 runs had them.
- Each run is one trajectory. Another `random_seed` gives another trajectory and
  somewhat different numbers.
- The MD takes most of the time. With MACE-OMAT-0-medium a run with 50
  snapshots took 3.5, 6.1 and 10.8 minutes for 4, 8 and 16 ps. Going from 50 to
  100 snapshots added a few seconds of force calculations and fitting.

With a force field the force calculations cost little next to the MD, so more
snapshots are cheap. In this test they did not remove the imaginary modes,
though. The default stays at 50, because with VASP each snapshot is a DFT
calculation of the whole supercell. For the MD length, 4 ps gave about the same
agreement with AIMD as 8 and 16 ps for KNaICl. We kept the default of 8 ps for
the other materials.

## Known limitations

By default the MD runs at constant volume (NVT). At every temperature it keeps
the volume of the starting structure, as in Section 1. A real crystal expands as
it heats up, and the expansion changes the phonons. At high temperature, use
`run_npt=True` as in Section 2. The NPT cell is the average over one MD run, so
it carries the noise of that run.

The three MACE potentials give no Born charges, so the non-analytical
correction is not applied here. A `born_maker` can add them, either
`ForceFieldDielectricMaker` with MACE-Field (see the force field docs of
atomate2) or a VASP `DielectricMaker`.

The trajectory check uses fixed limits. A Lindemann ratio above 0.15 is reported
as melting. A displacement from the reference of more than 1.5 times the
vibration around the mean positions is reported as a shift. These are rules of
thumb. The limit of 0.15 is the Lindemann ratio at melting of an fcc solid. The
ratio at melting differs between materials. We did not
check whether any of the trajectories reported as melted is a liquid. A warning
is raised and the fit is still done. The ratio is averaged over all atoms. It
can pass 0.15 when only the light atoms move far, for example Li in a
superionic conductor, while the rest of the crystal stays solid.

Every result above comes from one MD trajectory. Section 3 shows that runs with
different settings differ by up to 0.1 THz on average and by up to 0.3 THz in
the highest frequency. Another `random_seed` gives somewhat different numbers.